# API wrappers

The OpenWeatherMap offers REST endpoints for querying current weather, forecasts, historical data, etc. However, accessing this data directly via the REST API requires handling multiple API calls, query parameters, and response parsing. The pyowm library abstracts these complexities and provides useful built-in functionalities.

After signing in to OpenWeatherMap retrieve your api key at https://home.openweathermap.org/api_keys

You will also need to install the pyowm package: pip install pyowm 

In [17]:
!pip install pyowm

In [18]:
import requests
import pyowm
import json

api_key = '57abc33d377e1ca397227d271c0ea598'

## use case 1: managing API keys

In a raw rest API call you always have to manage credentials in each individual call. Wrappers usually store and manage the authentication for you

In [19]:
from pyowm import OWM

# 1. Initialize the OWM object with your API key (it remembers it for us!)
owm = OWM(api_key)

# 2. Open the weather manager (this is the "assistant" that will make our requests)
mgr = owm.weather_manager()

print("Weather manager successfully initialized! 🌤️")

Weather manager successfully initialized! 🌤️


In [20]:
#You can get current weather data by making a GET request to an endpoint like:

params = {
    'appid' : api_key
}

response = requests.get('https://api.openweathermap.org/data/2.5/weather?q=London', params = params)

json.loads(response.text)

#but for every call you make using GET from now on you do need to add the parameters, since the raw API does not manage authentication for you

{'coord': {'lon': -0.1257, 'lat': 51.5085},
 'weather': [{'id': 721,
   'main': 'Haze',
   'description': 'haze',
   'icon': '50n'}],
 'base': 'stations',
 'main': {'temp': 281.74,
  'feels_like': 280.65,
  'temp_min': 280.93,
  'temp_max': 283.14,
  'pressure': 1021,
  'humidity': 90,
  'sea_level': 1021,
  'grnd_level': 1017},
 'visibility': 4500,
 'wind': {'speed': 2.06, 'deg': 80},
 'clouds': {'all': 100},
 'dt': 1772996265,
 'sys': {'type': 2,
  'id': 2075535,
  'country': 'GB',
  'sunrise': 1772951456,
  'sunset': 1772992320},
 'timezone': 0,
 'id': 2643743,
 'name': 'London',
 'cod': 200}

Most wrappers (pyowm included) include some way of initializing a session with the authentication key that you then don't need to type again.

Initialize pyowm with the default configuration. Thenopen the weather manager

Check out a snippet here: https://pyowm.readthedocs.io/en/latest/v3/code-recipes.html#weather_data

## use case 2: Simplified calls

With the raw REST API, you'd have to build a URL manually, send the request, and parse the JSON response to get the current weather.

In [21]:
city = 'London'
url = f'http://api.openweathermap.org/data/2.5/weather?q={city}'

response = requests.get(url,params= params)
data = response.json()
temperature = data['main']['temp']
humidity = data['main']['humidity']
wind_speed = data['wind']['speed']

print(f"Temperature: {temperature}°C, Humidity: {humidity}%, Wind Speed: {wind_speed} m/s")

Temperature: 281.74°C, Humidity: 90%, Wind Speed: 2.06 m/s


Get the equivalent call as above for the city of London using the pyowm package

In [22]:
# 1. Ask the weather manager for London's current weather
observation = mgr.weather_at_place('London')
w = observation.weather

temperature = w.temperature('celsius')['temp'] 
humidity = w.humidity
wind_speed = w.wind()['speed']

print(f"Temperature: {temperature}°C, Humidity: {humidity}%, Wind Speed: {wind_speed} m/s")

Temperature: 8.59°C, Humidity: 90%, Wind Speed: 2.06 m/s


## use case 3: Combining and chaining calls

Wrappers often offer methods that make multiple calls to batch requests that make sense to batch. And often they offer methods that make sequences of calls that each returns information necessary to make the next call.

For example, to get a weather forecast for a specific city using the raw API you need to first geocode the city to get its latitude and longitude:

In [23]:
city = 'New York'
geocode_url = f'http://api.openweathermap.org/data/2.5/weather?q={city}'
geocode_response = requests.get(geocode_url,params=params).json()

lat = geocode_response['coord']['lat']
lon = geocode_response['coord']['lon']

Then, request the weather forecast for that latitude/longitude:

In [24]:
forecast_url = f'http://api.openweathermap.org/data/2.5/forecast?lat={lat}&lon={lon}'
forecast_response = requests.get(forecast_url, params=params).json()

for entry in forecast_response['list']:
    print(f"Time: {entry['dt_txt']}, Temp: {entry['main']['temp']}°C")

Time: 2026-03-08 21:00:00, Temp: 292.3°C
Time: 2026-03-09 00:00:00, Temp: 289.51°C
Time: 2026-03-09 03:00:00, Temp: 283.68°C
Time: 2026-03-09 06:00:00, Temp: 281.95°C
Time: 2026-03-09 09:00:00, Temp: 287.58°C
Time: 2026-03-09 12:00:00, Temp: 280.61°C
Time: 2026-03-09 15:00:00, Temp: 284.67°C
Time: 2026-03-09 18:00:00, Temp: 289.25°C
Time: 2026-03-09 21:00:00, Temp: 289.99°C
Time: 2026-03-10 00:00:00, Temp: 287.09°C
Time: 2026-03-10 03:00:00, Temp: 284.91°C
Time: 2026-03-10 06:00:00, Temp: 283.66°C
Time: 2026-03-10 09:00:00, Temp: 282.68°C
Time: 2026-03-10 12:00:00, Temp: 282.13°C
Time: 2026-03-10 15:00:00, Temp: 286.36°C
Time: 2026-03-10 18:00:00, Temp: 289.34°C
Time: 2026-03-10 21:00:00, Temp: 286.9°C
Time: 2026-03-11 00:00:00, Temp: 284.42°C
Time: 2026-03-11 03:00:00, Temp: 284.18°C
Time: 2026-03-11 06:00:00, Temp: 282.66°C
Time: 2026-03-11 09:00:00, Temp: 281.6°C
Time: 2026-03-11 12:00:00, Temp: 280.59°C
Time: 2026-03-11 15:00:00, Temp: 282.93°C
Time: 2026-03-11 18:00:00, Temp: 286.

Two calls: one for geocoding, one for forecasts.
But with pyowm, because this is a common operation, there is a method that handles the geocoding internally and then fetches the weather forecast in one step.

Get the above forecast in a single call using pyowm.

Hint: search for "forecast_at_place" in the code recipies of the documentation

In [25]:
forecast_data = mgr.forecast_at_place('New York', '3h').forecast

for weather in forecast_data:
    time = weather.reference_time('iso')
    temp = weather.temperature('celsius')['temp']
    print(f"Time: {time}, Temp: {temp}°C")

Time: 2026-03-08 21:00:00+00:00, Temp: 19.16°C
Time: 2026-03-09 00:00:00+00:00, Temp: 16.36°C
Time: 2026-03-09 03:00:00+00:00, Temp: 10.53°C
Time: 2026-03-09 06:00:00+00:00, Temp: 8.8°C
Time: 2026-03-09 09:00:00+00:00, Temp: 14.43°C
Time: 2026-03-09 12:00:00+00:00, Temp: 7.46°C
Time: 2026-03-09 15:00:00+00:00, Temp: 11.52°C
Time: 2026-03-09 18:00:00+00:00, Temp: 16.1°C
Time: 2026-03-09 21:00:00+00:00, Temp: 16.84°C
Time: 2026-03-10 00:00:00+00:00, Temp: 13.94°C
Time: 2026-03-10 03:00:00+00:00, Temp: 11.76°C
Time: 2026-03-10 06:00:00+00:00, Temp: 10.51°C
Time: 2026-03-10 09:00:00+00:00, Temp: 9.53°C
Time: 2026-03-10 12:00:00+00:00, Temp: 8.98°C
Time: 2026-03-10 15:00:00+00:00, Temp: 13.21°C
Time: 2026-03-10 18:00:00+00:00, Temp: 16.19°C
Time: 2026-03-10 21:00:00+00:00, Temp: 13.75°C
Time: 2026-03-11 00:00:00+00:00, Temp: 11.27°C
Time: 2026-03-11 03:00:00+00:00, Temp: 11.03°C
Time: 2026-03-11 06:00:00+00:00, Temp: 9.51°C
Time: 2026-03-11 09:00:00+00:00, Temp: 8.45°C
Time: 2026-03-11 12:0

## use case 4: Convenience methods

Wrappers often offer built-in methods to handle common kinds of tasks related to the APIs, reducing the need for manual calculations.

for example converting units (e.g., temperature from Celsius to Fahrenheit) or working with more complex data requires manual conversion when using the raw API.

In [26]:
city = 'London'
url = f'http://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}'

response = requests.get(url)
data = response.json()
temperature_celsius = data['main']['temp']
temperature_fahrenheit = (temperature_celsius * 9/5) + 32

print(f"Temperature in Celsius: {temperature_celsius}°C, Fahrenheit: {temperature_fahrenheit}°F")

Temperature in Celsius: 281.74°C, Fahrenheit: 539.132°F


But the pyowm wrapper offers built-in methods to handle these kinds of tasks, reducing the need for manual calculations.
Get the temperature both in Celcius and Farenheit using pyowm. Navigate the code recipes to figure out the inbuilt methods for this.

In [27]:
observation = mgr.weather_at_place('London')
w = observation.weather

temp_celsius = w.temperature('celsius')['temp']
temp_fahrenheit = w.temperature('fahrenheit')['temp']

print(f"Temperature in Celsius: {temp_celsius}°C, Fahrenheit: {temp_fahrenheit}°F")

Temperature in Celsius: 8.59°C, Fahrenheit: 47.46°F
